# Gate 3 — Tmall pha 1: A0 vs A1 multi-seed

## ⚠ CHI PHÍ — đọc trước khi bấm Run all

Tmall lớn hơn RetailRocket rất nhiều:

| | RetailRocket | Tmall | tỉ lệ |
|---|---:|---:|---:|
| số sequence train | 30,690 | **462,945** | **15.1×** |
| batch/epoch (bs=64) | 480 | **7,234** | 15.1× |
| độ dài TB | 15.7 | 31.2 | 2.0× |

RetailRocket chạy ~5 phút/epoch. Ngoại suy thô cho Tmall: **1.9–3.1 giờ mỗi epoch**.
Với 25 epoch như retail → **47–77 giờ/run**, mà pha 1 cần **6 run** → **280–460 giờ**.
**Không khả thi trên Colab Pro.**

Vì vậy notebook này:
1. **Đo thật** thời gian một epoch Tmall trước (cell 6), rồi in ra dự toán.
2. Chạy với **`EPOCH_CAP` cố định**, áp **y hệt** cho A0 và A1.

`EPOCH_CAP` không phá tính công bằng: hai nhánh nhận **cùng ngân sách huấn luyện**. Nó chỉ có
nghĩa là ta so sánh ở một ngân sách cố định thay vì so ở điểm hội tụ — điều này **phải được ghi
rõ trong bài**, và là cách làm hợp lệ, phổ biến khi compute bị giới hạn.

**Chưa chốt `EPOCH_CAP`. Chạy tới cell 6 rồi dừng, gửi số đo về để chốt.**

---

## Tiêu chí đã khoá TRƯỚC khi thấy số

Đọc từ `experiments/seeds.json` (đã commit trước mọi lần chạy — kiểm chứng được bằng git log):

- **Seed**: `[2020, 2021, 2022]`, dùng chung cho **cả A0 lẫn A1**. Không đổi sau khi thấy kết quả.
- **Metric chính**: `ndcg@10`.
- **Báo cáo**: mean ± std mỗi nhánh + **Welch's t-test** (hai mẫu độc lập, phương sai không đều).
- **GO khi VÀ CHỈ KHI cả hai**: (1) hiệu mean vượt **sàn nhiễu đo trên chính Tmall** ở pha này,
  và (2) **p < 0.05**.

**σ của RetailRocket (≈0.00088) chỉ dùng để ước lượng số seed — KHÔNG đưa vào bài, KHÔNG dùng
làm sàn nhiễu của Tmall.** Sàn nhiễu Tmall tính từ chính 3 run A0 ở đây.

### Nếu hiệu A1−A0 nằm gọn trong nhiễu

**Đó là kết quả thật, không phải lỗi.** Nó nói: B1 không dịch chuyển metric này vượt quá mức mà
chạy lại chính model gốc đã dịch chuyển. Ghi nhận và báo cáo đúng như vậy — không chạy thêm seed
để "tìm" kết quả đẹp hơn.

## 1. GPU + Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## 2. Đường dẫn + clone code

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/DeAnThS'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'dataset')
CKPT_DIR     = os.path.join(PROJECT_ROOT, 'checkpoints')
LOG_DIR      = os.path.join(PROJECT_ROOT, 'logs')
RUN_META_DIR = os.path.join(PROJECT_ROOT, 'run_meta')
CODE_DIR     = '/content/MBHT-KDD22'
for d in [DATA_DIR, CKPT_DIR, LOG_DIR, RUN_META_DIR]:
    os.makedirs(d, exist_ok=True)

DATASET_ROOT = os.path.join(DATA_DIR, 'MBHT_dataset')
assert os.path.isdir(DATASET_ROOT), f'Chua co {DATASET_ROOT} -- chay notebook Gate 1-2 truoc.'

REPO_URL      = 'https://github.com/nguyenlmhcm/MBHT-KDD22.git'
BRANCH        = 'bt-mbht'
PINNED_COMMIT = '2755f88be51405d2e2d3227248ecb4a3fb5c6adb'
DATASET_NAME  = 'tmall_beh'
print('dataset:', DATASET_NAME)

In [ ]:
import subprocess, time

# Cell cai dat ben duoi co `%cd {CODE_DIR}`; buoc ra truoc de chay lai cell nay luon an toan.
os.chdir('/content')
if os.path.isdir(CODE_DIR):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)

for attempt in range(1, 6):
    r = subprocess.run(['git','clone','-b',BRANCH,REPO_URL,CODE_DIR], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'Clone OK (attempt {attempt})'); break
    print(f'attempt {attempt} failed:', r.stderr.strip())
    if os.path.isdir(CODE_DIR):
        subprocess.run(['rm','-rf',CODE_DIR], check=True)
    if attempt == 5:
        raise RuntimeError('git clone failed 5x')
    time.sleep(10*attempt)

subprocess.run(['git','-C',CODE_DIR,'checkout',PINNED_COMMIT], check=True)
head = subprocess.run(['git','-C',CODE_DIR,'rev-parse','HEAD'],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head.startswith(PINNED_COMMIT), f'{head} != {PINNED_COMMIT}'
print('commit:', head)

In [ ]:
%cd {CODE_DIR}
!pip install -q hyperopt pandas tqdm scikit_learn pyyaml colorlog colorama tensorboard

## 3. Đọc tiêu chí đã khoá từ repo

Seed **không** hardcode trong notebook — đọc từ file đã commit, nên có thể kiểm chứng bằng
`git log` rằng chúng được chốt trước khi chạy.

In [ ]:
import json

with open(os.path.join(CODE_DIR, 'experiments', 'seeds.json')) as f:
    LOCK = json.load(f)

SEEDS          = LOCK['seeds']
PRIMARY        = LOCK['primary_metric']
ALPHA          = LOCK['reporting']['alpha']

lock_commit = subprocess.run(
    ['git','-C',CODE_DIR,'log','-1','--format=%h %ci','--','experiments/seeds.json'],
    capture_output=True, text=True, check=True).stdout.strip()

print('seeds        :', SEEDS)
print('primary      :', PRIMARY)
print('test         :', LOCK['reporting']['test'])
print('alpha        :', ALPHA)
print('locked_at    :', LOCK['locked_at'])
print('seeds.json committed at:', lock_commit)
print()
for line in LOCK['go_criterion']:
    print(' ', line)

assert LOCK['locked_before_any_multiseed_run'] is True
assert LOCK['noise_floor'].get('tmall_beh') is None, \
    'Tmall noise floor da co san -- pha nay phai tu do lai tu chinh no'
print('\nOK: chua co san nhieu Tmall, se do tu 3 run A0 cua pha nay.')

## 4. Load Tmall + xác nhận |T| = 43 (suy ra, không hardcode)

In [ ]:
from recbole.config import Config
from recbole.data import create_dataset
from recbole.data.utils import get_dataloader, create_samplers
from recbole.model.transition_utils import transition_vocab_size

# Config Tmall lay dung tu run_MBHT.py: scales [10,4,20] (retail moi la [5,4,20])
FROZEN = {
    'USER_ID_FIELD': 'session_id', 'load_col': None, 'neg_sampling': None,
    'benchmark_filename': ['train', 'test'], 'alias_of_item_id': ['item_id_list'],
    'topk': [5, 10, 101], 'metrics': ['Recall', 'NDCG', 'MRR'],
    'valid_metric': 'NDCG@10', 'eval_args': {'mode': 'full', 'order': 'TO'},
    'MAX_ITEM_LIST_LENGTH': 200,
    'train_batch_size': 64, 'eval_batch_size': 128,
    'hyper_len': 6, 'scales': [10, 4, 20],
    'enable_hg': 1, 'enable_ms': 1, 'customized_eval': 1, 'abaltion': '',
}

base_cfg = {**FROZEN, 'data_path': DATASET_ROOT}
schema_config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=base_cfg)
dataset = create_dataset(schema_config)

type_map = dataset.field2token_id['item_type_list']
n_types  = len(type_map)
T        = transition_vocab_size(n_types)
print('field2token_id[item_type_list] =', type_map)
print('n_types =', n_types, '  |T| =', T)

# Tmall khac RetailRocket: 6 type -> |T|=43, khong phai 31.
assert n_types == 6, f'Tmall phai co 6 type, dang thay {n_types}'
assert T == 43, f'|T| phai la 43, dang la {T}'
EXPECTED_PARAM_DELTA = T * 64
print('param delta ky vong =', EXPECTED_PARAM_DELTA, '(retail la 1984 -- khac nhau la dung)')


def fresh_split(cfg):
    """Mot ban du lieu sach cho MOT consumer. Khong bao gio dung chung:
    build() doi inter_feat tai cho (goi lan hai la crash), va train dataloader
    shuffle() split tai cho moi epoch nen run sau se khoi dau tu thu tu run
    truoc de lai."""
    ds = create_dataset(cfg)
    tr, te = ds.build()
    return ds, tr, te

## 5. ĐO CHI PHÍ THẬT — dừng ở đây, gửi số về trước khi chạy tiếp

Đo thời gian **một epoch** Tmall rồi ngoại suy. Cell này train đúng 1 epoch rồi bỏ,
không lưu checkpoint, không ảnh hưởng kết quả sau.

In [ ]:
import time as _time
from logging import getLogger
from recbole.model.sequential_recommender.mbht import MBHT
from recbole.utils import init_logger, init_seed, get_trainer

probe_cfg = Config(model='MBHT', dataset=DATASET_NAME, config_dict={
    **base_cfg, 'seed': SEEDS[0], 'gpu_id': 0,
    'enable_transition_embedding': 0,
    'epochs': 1, 'checkpoint_dir': '/content/_probe_ckpt'})
os.makedirs('/content/_probe_ckpt', exist_ok=True)
init_seed(probe_cfg['seed'], probe_cfg['reproducibility'])

p_ds, p_tr, p_te = fresh_split(probe_cfg)
s1, s2 = create_samplers(probe_cfg, p_ds, [p_tr, p_te])
p_train = get_dataloader(probe_cfg, 'train')(probe_cfg, p_tr, s1, shuffle=True)
p_test  = get_dataloader(probe_cfg, 'test')(probe_cfg, p_te, s2, shuffle=False)

print('batches/epoch:', len(p_train))
p_model = MBHT(probe_cfg, p_train.dataset).to(probe_cfg['device'])
p_trainer = get_trainer(probe_cfg['MODEL_TYPE'], probe_cfg['model'])(probe_cfg, p_model)

t0 = _time.time()
p_trainer._train_epoch(p_train, 0, show_progress=True)
epoch_s = _time.time() - t0

t0 = _time.time()
p_trainer._valid_epoch(p_test, show_progress=False)
eval_s = _time.time() - t0

per_run = lambda n_ep: (epoch_s * n_ep + eval_s * n_ep) / 3600.0
print(f'\n=== DO THUC TE (Tmall, batch_size 64) ===')
print(f'  1 epoch train : {epoch_s/60:.1f} phut')
print(f'  1 lan eval    : {eval_s/60:.1f} phut')
print(f'\n{"EPOCH_CAP":>10} {"gio/run":>10} {"gio 6 run":>12}')
for n_ep in (1, 2, 3, 5, 8, 10, 15, 25):
    print(f'{n_ep:>10} {per_run(n_ep):>10.1f} {per_run(n_ep)*6:>12.1f}')
print('\n6 run = A0 x 3 seed + A1 x 3 seed.')
print('DUNG LAI. Gui bang nay ve de chot EPOCH_CAP truoc khi chay tiep.')

del p_model, p_trainer, p_train, p_test, p_ds, p_tr, p_te

## 6. Chốt EPOCH_CAP rồi mới chạy tiếp

Điền `EPOCH_CAP` bằng con số đã thống nhất. Giá trị này áp **y hệt** cho A0 và A1 —
đó là điều giữ cho so sánh công bằng.

In [ ]:
EPOCH_CAP = None   # <-- DIEN SAU KHI CHOT. Vi du: EPOCH_CAP = 5

assert EPOCH_CAP is not None, 'Chua chot EPOCH_CAP -- xem bang do o cell truoc'
assert isinstance(EPOCH_CAP, int) and EPOCH_CAP > 0
print('EPOCH_CAP =', EPOCH_CAP, '(ap dung y het cho A0 va A1)')

## 7. Hàm chạy một run (lưu kết quả ra Drive để sống sót qua timeout)

In [ ]:
import glob, hashlib
from recbole.utils import set_color

def run(arm, seed):
    """arm: 'A0' hoac 'A1'. Tra ve dict ket qua; bo qua neu da chay xong."""
    enable_b1 = (arm == 'A1')
    tag  = f'{arm}-{DATASET_NAME}-seed{seed}-cap{EPOCH_CAP}'
    meta = os.path.join(RUN_META_DIR, f'{tag}.json')

    if os.path.exists(meta):
        with open(meta) as f:
            done = json.load(f)
        print(f'[{tag}] da co ket qua, bo qua')
        return done

    ckpt = os.path.join(CKPT_DIR, tag); os.makedirs(ckpt, exist_ok=True)
    logd = os.path.join(LOG_DIR, tag);  os.makedirs(logd, exist_ok=True)

    cfg = Config(model='MBHT', dataset=DATASET_NAME, config_dict={
        **base_cfg, 'checkpoint_dir': ckpt, 'gpu_id': 0, 'seed': seed,
        'epochs': EPOCH_CAP, 'enable_transition_embedding': int(enable_b1)})
    init_seed(cfg['seed'], cfg['reproducibility'])
    init_logger(cfg, log_root=logd)
    getLogger().info(f'TAG={tag} commit={PINNED_COMMIT} arm={arm} seed={seed} cap={EPOCH_CAP}')

    ds, tr, te = fresh_split(cfg)
    s_tr, s_te = create_samplers(cfg, ds, [tr, te])
    train_data = get_dataloader(cfg, 'train')(cfg, tr, s_tr, shuffle=True)
    test_data  = get_dataloader(cfg, 'test')(cfg, te, s_te, shuffle=False)

    model = MBHT(cfg, train_data.dataset).to(cfg['device'])
    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{tag}] |T|={model.n_transitions} n_types={model.n_behavior_types} params={n_params}')

    trainer = get_trainer(cfg['MODEL_TYPE'], cfg['model'])(cfg, model)
    prev = sorted(glob.glob(os.path.join(ckpt, '*.pth')), key=os.path.getmtime)
    if prev:
        print(f'[{tag}] resuming from {prev[-1]}')
        trainer.resume_checkpoint(prev[-1])

    _, result = trainer.fit(train_data, test_data, saved=True,
                            show_progress=cfg['show_progress'])
    if result is None:
        raise RuntimeError(
            f'[{tag}] khong co ket qua: resume tu checkpoint da train xong nen fit() '
            f'bo qua vong epoch. Xoa {ckpt} roi chay lai.')

    h = hashlib.sha256()
    for k, v in sorted(model.state_dict().items()):
        h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())

    out = {'tag': tag, 'arm': arm, 'seed': seed, 'epoch_cap': EPOCH_CAP,
           'commit': PINNED_COMMIT, 'n_params': n_params,
           'param_hash': h.hexdigest()[:16],
           'result': {k: float(v) for k, v in result.items()}}
    with open(meta, 'w') as f:
        json.dump(out, f, indent=2)
    print(set_color(f'[{tag}] ', 'yellow') + json.dumps(out['result']))
    return out

## 8. PASS #2 — param delta phải đúng 2,752 (không phải 1,984 của retail)

In [ ]:
import torch

c_off = Config(model='MBHT', dataset=DATASET_NAME,
               config_dict={**base_cfg, 'seed': SEEDS[0], 'enable_transition_embedding': 0})
c_on  = Config(model='MBHT', dataset=DATASET_NAME,
               config_dict={**base_cfg, 'seed': SEEDS[0], 'enable_transition_embedding': 1})
pd_, ptr, pte = fresh_split(c_off)
ps, _ = create_samplers(c_off, pd_, [ptr, pte])
probe = get_dataloader(c_off, 'train')(c_off, ptr, ps, shuffle=False)

init_seed(SEEDS[0], True); m_off = MBHT(c_off, probe.dataset)
init_seed(SEEDS[0], True); m_on  = MBHT(c_on,  probe.dataset)
d = sum(p.numel() for p in m_on.parameters()) - sum(p.numel() for p in m_off.parameters())
print(f'param delta = {d}   (ky vong {EXPECTED_PARAM_DELTA} = |T| 43 x 64)')
assert d == EXPECTED_PARAM_DELTA, f'PASS #2 FAIL: {d} != {EXPECTED_PARAM_DELTA}'

sd_off, sd_on = m_off.state_dict(), m_on.state_dict()
assert set(sd_on) - set(sd_off) == {'transition_embedding.weight'}
bad = [k for k in sd_off if not torch.equal(sd_off[k], sd_on[k])]
assert not bad, f'bat B1 lam lech tham so A0: {bad}'
print('PASS #2 OK')
del m_off, m_on, probe, pd_, ptr, pte

## 9. Chạy A0 × 3 seed, rồi A1 × 3 seed

Mỗi run tự lưu ra `run_meta/`. Nếu Colab ngắt giữa chừng, chạy lại cell này — các run đã
xong sẽ được bỏ qua, run dở sẽ resume từ checkpoint.

In [ ]:
results = {'A0': [], 'A1': []}
for arm in ('A0', 'A1'):
    for seed in SEEDS:
        results[arm].append(run(arm, seed))
        print()
print('Xong:', {a: len(v) for a, v in results.items()})

## 10. Phân tích — mean±std, sàn nhiễu Tmall, Welch's t-test

In [ ]:
import statistics
from scipy import stats

vals = {arm: [r['result'][PRIMARY] for r in results[arm]] for arm in ('A0','A1')}

# San nhieu Tmall: do TU CHINH 3 run A0 cua pha nay (khong dung so cua RetailRocket)
noise_tmall = max(vals['A0']) - min(vals['A0'])

print(f'=== {PRIMARY} tren {DATASET_NAME}, EPOCH_CAP={EPOCH_CAP}, seeds={SEEDS} ===\n')
for arm in ('A0','A1'):
    v = vals[arm]
    print(f'{arm}: ' + '  '.join(f'{x:.4f}' for x in v) +
          f'   mean={statistics.mean(v):.4f}  std={statistics.stdev(v):.4f}')

diff = statistics.mean(vals['A1']) - statistics.mean(vals['A0'])
t, p = stats.ttest_ind(vals['A1'], vals['A0'], equal_var=False)   # Welch

print(f'\nsan nhieu Tmall (range 3 run A0) : {noise_tmall:.4f}')
print(f'hieu A1 - A0                     : {diff:+.4f}')
print(f"Welch's t-test                   : t={t:+.3f}  p={p:.4f}  (alpha={ALPHA})")

c1 = abs(diff) > noise_tmall
c2 = p < ALPHA
print(f'\n  (1) hieu vuot san nhieu ? {c1}')
print(f'  (2) p < {ALPHA} ?           {c2}')
print('\n' + '='*66)
if c1 and c2:
    print(f'GO -- B1 {"tang" if diff>0 else "GIAM"} {PRIMARY} vuot ca nhieu lan y nghia thong ke.')
    if diff < 0:
        print('Luu y: huong AM. B1 lam giam metric mot cach co y nghia.')
else:
    print('TRONG NHIEU / KHONG CO Y NGHIA -- day la KET QUA THAT, khong phai loi.')
    print('B1 khong dich chuyen metric nay vuot muc ma chay lai chinh model goc da dich chuyen.')
    print('Khong chay them seed de "tim" ket qua dep hon. Neu muon mo rong, phai khai bao')
    print('seeds_phase2 trong experiments/seeds.json TRUOC, commit, roi bao ca hai pha.')
print('='*66)

summary = {'dataset': DATASET_NAME, 'epoch_cap': EPOCH_CAP, 'seeds': SEEDS,
           'primary_metric': PRIMARY, 'values': vals,
           'mean': {a: statistics.mean(v) for a, v in vals.items()},
           'std': {a: statistics.stdev(v) for a, v in vals.items()},
           'noise_floor_tmall': noise_tmall, 'diff': diff,
           'welch_t': float(t), 'welch_p': float(p), 'alpha': ALPHA,
           'exceeds_noise': bool(c1), 'significant': bool(c2),
           'verdict': 'GO' if (c1 and c2) else 'WITHIN_NOISE_OR_NS'}
with open(os.path.join(RUN_META_DIR, f'phase1_summary_{DATASET_NAME}.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('\nDa luu summary:', os.path.join(RUN_META_DIR, f'phase1_summary_{DATASET_NAME}.json'))

## 11. Bảng đầy đủ mọi metric (tham khảo — quyết định chỉ dựa trên metric chính)

In [ ]:
all_keys = sorted(results['A0'][0]['result'])
print(f'{"metric":>12} {"A0 mean":>10} {"A0 std":>9} {"A1 mean":>10} {"A1 std":>9} {"diff":>9} {"p":>8}')
for k in all_keys:
    a0 = [r['result'][k] for r in results['A0']]
    a1 = [r['result'][k] for r in results['A1']]
    if statistics.stdev(a0) == 0 and statistics.stdev(a1) == 0:
        pk = float('nan')
    else:
        pk = stats.ttest_ind(a1, a0, equal_var=False).pvalue
    star = ' *' if k == PRIMARY else ''
    print(f'{k:>12} {statistics.mean(a0):>10.4f} {statistics.stdev(a0):>9.4f} '
          f'{statistics.mean(a1):>10.4f} {statistics.stdev(a1):>9.4f} '
          f'{statistics.mean(a1)-statistics.mean(a0):>+9.4f} {pk:>8.4f}{star}')
print('\n* = metric chinh da khoa truoc. Cac dong khac chi de tham khao;')
print('  doc nhieu metric roi chon cai dep nhat la multiple-comparison, reviewer se bat.')

## 12. Cần dán về

1. Bảng đo chi phí ở cell 6 (**gửi trước, chờ chốt EPOCH_CAP**).
2. PASS #2 — param delta = 2,752.
3. Bảng mean±std + Welch's t-test ở cell 10.
4. Bảng đầy đủ ở cell 11.
5. Bất kỳ NaN/traceback nào.

Nếu Colab ngắt giữa chừng: chạy lại cell 9, các run đã xong tự bỏ qua.